# Model building & verification — per-pair covariate-conditioned distributions

For each kept pair, fit its finalized distribution with the AFT scale depending on the
finalized covariates, using **forward selection** (a covariate is retained only if it adds
ΔAIC ≥ 2 *conditional on* those already in), then **verify** with Cox–Snell residuals
(which must look Exponential(1) for a correct model).

Protections against the flexible-distribution artifacts seen earlier:
* `loc` is estimated (shared) so the physical minimum headway is respected;
* if the primary distribution's final model **fails Cox–Snell (KS p < 0.05)** or does not
  converge, the pair is **automatically refit under Weibull** (stable) and re-selected;
* forward selection re-tests every covariate jointly, dropping univariate artifacts.

Outputs: fitted coefficients (with SE / z / p), goodness-of-fit + model comparison, and a
Cox–Snell Q–Q chart per pair (native Excel, no matplotlib).

In [1]:
# --- Cell 1: Imports, paths, finalized per-pair spec ---
import os, warnings
import numpy as np
import pandas as pd
from scipy import stats, optimize
warnings.simplefilter("ignore")

BASE      = r"D:\Headway"
DATA_PATH = os.path.join(BASE, "data3.xlsx")
TABLES    = os.path.join(BASE, "Tables")
os.makedirs(TABLES, exist_ok=True)

OUTCOME  = "Time_Headway"
ALPHA    = 0.05
DAIC_MIN = 2.0
FALLBACK = "weibull_min"     # stable distribution used when the primary model fails verification

# finalized (distribution, candidate covariates) per pair  -- from screening/VIF (Sheet3)
FINAL_SPEC = {
    "BTW_following_4W":     ("weibull_min", ["speed_diff", "speed", "flow"]),
    "BTW_following_MT_3W":  ("weibull_min", ["speed", "speed_diff", "flow", "occupancy"]),
    "BTW_following_NMT_3W": ("gengamma",    ["speed", "flow"]),
    "PR_following_MT_3W":   ("weibull_min", ["speed", "occupancy", "flow"]),
    "BTW_following_MT_2W":  ("gengamma",    ["speed_diff", "speed"]),
    "PR_following_NMT_3W":  ("gengamma",    ["occupancy", "flow"]),
    "PR_following_4W":      ("gengamma",    []),
    "BTW_following_NMT_2W": ("gengamma",    ["speed_diff"]),
}

# covariate -> (column, type, unit, label) for design + reporting
COV = {
    "speed":      ("Target_Speed_km/hr", "cont", 1,    "per +1 km/h"),
    "speed_diff": ("Speed_Difference",   "cont", 1,    "per +1 km/h"),
    "flow":       ("Flow_pcu/hr",        "cont", 1000, "per +1000 pcu/hr"),
    "occupancy":  ("_occ01",             "bin",  1,    "True vs False"),
    "off_cen":    ("_offc01",            "bin",  1,    "True vs False"),
}

In [2]:
# --- Cell 2: Load data3 and encode covariates ---
df = pd.read_excel(DATA_PATH)
SPEED_COL = "Target_Speed_km/hr" if "Target_Speed_km/hr" in df.columns else "Subject_Speed_km/hr"
COV["speed"] = (SPEED_COL,) + COV["speed"][1:]
df["_occ01"]  = df["Occupancy"].astype(int)
df["_offc01"] = df["Off_centeredness"].astype(int)
print("rows:", len(df), "| pairs modelled:", list(FINAL_SPEC.keys()))

rows: 898 | pairs modelled: ['BTW_following_4W', 'BTW_following_MT_3W', 'BTW_following_NMT_3W', 'PR_following_MT_3W', 'BTW_following_MT_2W', 'PR_following_NMT_3W', 'PR_following_4W', 'BTW_following_NMT_2W']


In [3]:
# --- Cell 3: AFT machinery (free loc, robust fit, forward selection, Cox-Snell) ---
def _design(g, covs):
    if not covs: return np.empty((len(g), 0))
    cols = []
    for c in covs:
        col, typ = COV[c][0], COV[c][1]
        v = pd.to_numeric(g[col], errors="coerce").values.astype(float)
        cols.append(v - v.mean() if typ == "cont" else v)
    return np.column_stack(cols)

def _nll(p, t, Z, dist, ns):
    # shapes are log-transformed (kept positive & stable); loc fixed at 0
    sh = [np.exp(p[i]) for i in range(ns)]
    lin = p[ns] + (Z @ p[ns+1:] if Z.shape[1] else 0.0)
    lp = dist.logpdf(t, *sh, loc=0, scale=np.exp(lin))
    return -lp.sum() if np.all(np.isfinite(lp)) else 1e12

def fit_aft(t, Z, dist_name):
    dist = getattr(stats, dist_name); ns = dist.numargs
    init = dist.fit(t, floc=0)
    x0 = [np.log(max(v, 1e-3)) for v in init[:ns]] + [np.log(init[-1])] + [0.0]*Z.shape[1]
    best = None
    for method, opt in (("Nelder-Mead", {"maxiter":20000,"xatol":1e-9,"fatol":1e-9}),
                        ("Powell",       {"maxiter":20000})):
        for jit in range(3):
            xx = np.array(x0) + (np.random.RandomState(jit).normal(0, 0.1, len(x0)) if jit else 0.0)
            r = optimize.minimize(_nll, xx, args=(t, Z, dist, ns), method=method, options=opt)
            if best is None or r.fun < best.fun: best = r
    return best, ns, dist

def cox_snell(t, Z, res):
    r, ns, dist = res; p = r.x; sh = [np.exp(p[i]) for i in range(ns)]
    lin = p[ns] + (Z @ p[ns+1:] if Z.shape[1] else 0.0)
    F = np.clip(dist.cdf(t, *sh, loc=0, scale=np.exp(lin)), 1e-12, 1-1e-12)
    cs = -np.log(1 - F)
    D, pks = stats.kstest(cs, "expon")
    n = len(cs); q = stats.expon.ppf((np.arange(1, n+1) - 0.5)/n)
    return cs, round(D, 4), pks, round(np.corrcoef(np.sort(cs), q)[0, 1], 4)

def numeric_se(fn, x, args, eps=1e-4):
    n = len(x); H = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            xa = x.copy(); xa[i]+=eps; xa[j]+=eps
            xb = x.copy(); xb[i]+=eps; xb[j]-=eps
            xc = x.copy(); xc[i]-=eps; xc[j]+=eps
            xd = x.copy(); xd[i]-=eps; xd[j]-=eps
            H[i, j] = (fn(xa,*args)-fn(xb,*args)-fn(xc,*args)+fn(xd,*args))/(4*eps*eps)
    try:    cov = np.linalg.inv(H)
    except np.linalg.LinAlgError: cov = np.linalg.pinv(H)
    with np.errstate(invalid="ignore"):
        return np.sqrt(np.abs(np.diag(cov)))

def forward_select(g, t, dist_name, candidates):
    """Greedily add covariates while each adds conditional dAIC >= DAIC_MIN and p<ALPHA."""
    r0 = fit_aft(t, _design(g, []), dist_name); ll0 = -r0[0].fun; k0 = r0[1] + 1
    selected, ll_cur, k_cur = [], ll0, k0
    remaining = list(candidates)
    while remaining:
        best = None
        for cov in remaining:
            trial = selected + [cov]
            rt = fit_aft(t, _design(g, trial), dist_name); llt = -rt[0].fun
            dAIC = (2*k_cur - 2*ll_cur) - (2*(k_cur+1) - 2*llt)
            p = stats.chi2.sf(2*(llt - ll_cur), 1)
            if (dAIC >= DAIC_MIN and p < ALPHA) and (best is None or dAIC > best[1]):
                best = (cov, dAIC, llt)
        if best is None: break
        selected.append(best[0]); remaining.remove(best[0])
        ll_cur, k_cur = best[2], k_cur + 1
    return selected

In [4]:
# --- Cell 4: Build + verify each pair's model (with Weibull fallback) ---
coef_rows, gof_rows, cs_store = [], [], {}
for pair, (dist0, cands) in FINAL_SPEC.items():
    g = df[df[OUTCOME].notna() & (df["Pair"] == pair)]
    t = g[OUTCOME].values; n = len(t)

    def build(dist_name):
        sel = forward_select(g, t, dist_name, cands)
        Z = _design(g, sel); res = fit_aft(t, Z, dist_name)
        cs, D, pks, qq = cox_snell(t, Z, res)
        return sel, Z, res, cs, D, pks, qq

    dist_used = dist0
    sel, Z, res, cs, D, pks, qq = build(dist0)
    note = "primary"
    # fallback if verification fails or non-convergence, and primary isn't already Weibull
    if (pks < ALPHA or not res[0].success) and dist0 != FALLBACK:
        dist_used = FALLBACK
        sel, Z, res, cs, D, pks, qq = build(FALLBACK)
        note = f"fell back to {FALLBACK} (primary {dist0} failed verification)"

    r, ns, dist = res; params = r.x
    ll = -r.fun; k = ns + 1 + len(sel)
    r0 = fit_aft(t, _design(g, []), dist_used); ll0 = -r0[0].fun; k0 = r0[1] + 1
    LR = 2*(ll - ll0); pj = stats.chi2.sf(LR, len(sel)) if sel else np.nan
    se = numeric_se(_nll, params, (t, Z, dist, ns))

    # coefficient rows (scale covariates only)
    for i, cov in enumerate(sel):
        b = params[ns+1+i]; s = se[ns+1+i]; z = b/s if s>0 else np.nan
        unit, lab = COV[cov][2], COV[cov][3]
        eff = ((np.exp(b*unit)-1)*100)
        coef_rows.append(dict(Pair=pair, N=n, Distribution=dist_used, Covariate=cov,
                              coef=round(b,5), SE=round(s,5), z=round(z,2),
                              p="<0.001" if (pd.notna(z) and 2*stats.norm.sf(abs(z))<1e-3) else
                                (round(2*stats.norm.sf(abs(z)),4) if pd.notna(z) else np.nan),
                              effect=f"{eff:+.2f}%  {lab}"))
    if not sel:
        coef_rows.append(dict(Pair=pair, N=n, Distribution=dist_used, Covariate="(none - decoupled)",
                              coef=np.nan, SE=np.nan, z=np.nan, p=np.nan, effect="plain distribution"))

    gof_rows.append(dict(Pair=pair, N=n, Distribution=dist_used,
                         Final_covariates=", ".join(sel) if sel else "none (decoupled)",
                         joint_LR=round(LR,2), joint_p="<0.001" if (pd.notna(pj) and pj<1e-3) else
                                  (round(pj,4) if pd.notna(pj) else "n/a"),
                         dAIC_vs_null=round((2*k0-2*ll0)-(2*k-2*ll),2),
                         CoxSnell_KS_D=D, CoxSnell_KS_p="<0.001" if pks<1e-3 else round(pks,3),
                         QQ_corr=qq, fit_ok="Yes" if pks>=ALPHA else "NO", note=note,
                         converged=bool(r.success)))
    cs_store[pair] = cs

coefficients = pd.DataFrame(coef_rows)
goodness     = pd.DataFrame(gof_rows)
goodness

,Pair,N,Distribution,Final_covariates,joint_LR,joint_p,dAIC_vs_null,CoxSnell_KS_D,CoxSnell_KS_p,QQ_corr,fit_ok,note,converged
0,BTW_following_4W,250,weibull_min,"speed_diff, speed",59.43,<0.001,55.43,0.0471,0.617,0.9956,Yes,primary,True
1,BTW_following_MT_3W,186,weibull_min,"speed, speed_diff",50.29,<0.001,46.29,0.1074,0.025,0.9819,NO,primary,True
2,BTW_following_NMT_3W,119,gengamma,speed,5.98,0.0145,3.98,0.0501,0.911,0.9887,Yes,primary,True
3,PR_following_MT_3W,104,weibull_min,"speed, occupancy",25.51,<0.001,21.51,0.0689,0.681,0.9914,Yes,primary,True
4,BTW_following_MT_2W,73,gengamma,"speed_diff, speed",13.48,0.0012,9.48,0.0874,0.602,0.9878,Yes,primary,True
5,PR_following_NMT_3W,49,weibull_min,none (decoupled),0.00,n/a,0.00,0.0912,0.776,0.9830,Yes,fell back to weibull_min (primary gengamma fai...,True
6,PR_following_4W,43,gengamma,none (decoupled),0.00,n/a,0.00,0.1137,0.594,0.9864,Yes,primary,True
7,BTW_following_NMT_2W,41,weibull_min,speed_diff,8.57,0.0034,6.57,0.1192,0.564,0.9640,Yes,fell back to weibull_min (primary gengamma fai...,True


In [ ]:
# --- Cell 5: Save coefficients + goodness-of-fit -> Excel ---
out_path = os.path.join(TABLES, "06_model_fit_verification.xlsx")
with pd.ExcelWriter(out_path, engine="openpyxl") as xl:
    goodness.to_excel(xl,     sheet_name="Model_fit_GoF", index=False)
    coefficients.to_excel(xl, sheet_name="Coefficients", index=False)
print("Saved:", out_path)
coefficients

In [ ]:
# --- Cell 6: Cox-Snell Q-Q chart per pair (native Excel, no matplotlib) ---
from openpyxl import load_workbook
from openpyxl.chart import ScatterChart, Reference, Series
wb = load_workbook(out_path)
for pair, cs in cs_store.items():
    cs_sorted = np.sort(cs); n = len(cs_sorted)
    theo = stats.expon.ppf((np.arange(1, n+1) - 0.5)/n)   # exponential(1) quantiles
    ws = wb.create_sheet(("qq_" + pair)[:31].replace("/", "_"))
    ws["A1"], ws["B1"], ws["D1"], ws["E1"] = "theoretical", "cox_snell", "ref_x", "ref_y"
    for i in range(n):
        ws.cell(row=i+2, column=1, value=float(theo[i]))
        ws.cell(row=i+2, column=2, value=float(cs_sorted[i]))
    mx = float(max(theo.max(), cs_sorted.max()))
    ws["D2"], ws["E2"], ws["D3"], ws["E3"] = 0, 0, mx, mx   # 45-degree reference line
    ch = ScatterChart(); ch.title = f"Cox-Snell Q-Q: {pair}"
    ch.x_axis.title = "Exponential(1) quantile"; ch.y_axis.title = "Cox-Snell residual"
    ch.x_axis.delete = False; ch.y_axis.delete = False
    s_pts = Series(Reference(ws, min_col=2, min_row=1, max_row=n+1),
                   Reference(ws, min_col=1, min_row=2, max_row=n+1), title_from_data=True)
    s_pts.marker.symbol = "circle"; s_pts.graphicalProperties.line.noFill = True
    s_ref = Series(Reference(ws, min_col=5, min_row=1, max_row=3),
                   Reference(ws, min_col=4, min_row=2, max_row=3), title_from_data=True)
    ch.series.append(s_pts); ch.series.append(s_ref)
    ch.height, ch.width = 9, 12; ws.add_chart(ch, "G2")
wb.save(out_path)
print("Embedded Cox-Snell Q-Q charts into:", out_path)